<a href="https://colab.research.google.com/github/Jeffrey1999/Deep-Learning/blob/main/Segmentation_based_Phishing_URL_Detection_Eint_Sandi_Aung%2C_E%2C_and_Aung.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install wordsegment

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 47.3 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
from transformers import BertTokenizer
from wordsegment import load, segment
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding, Dropout
from tensorflow.keras.preprocessing.sequence import pad_sequences
from urllib.parse import urlparse
import re

In [ ]:

# Initialize wordsegment and BERT tokenizer
load()  # Load wordsegment dictionary
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [ ]:
# Function to preprocess and tokenize URLs (inspired by URL-Tokenizer/SegURLizer)
def tokenize_url(url):
    # Parse URL components
    parsed = urlparse(url)
    domain = parsed.netloc.lower()
    path = parsed.path.lower()

    # Remove common prefixes/suffixes
    domain = re.sub(r'^(www\.|http://|https://)', '', domain)

    # Word-level segmentation using wordsegment
    domain_segments = segment(domain.replace('.', ''))
    path_segments = segment(path.replace('/', '')) if path else []
    # BERT tokenization (character-level and subword)
    bert_tokens = bert_tokenizer.tokenize(domain + ' ' + path)

    # Combine segmented tokens and BERT tokens
    combined_tokens = domain_segments + path_segments + bert_tokens

    # Return combined tokens
    return combined_tokens

In [ ]:
# Function to extract basic NLP features (inspired by PhiSN)
def extract_nlp_features(url):
    parsed = urlparse(url)
    domain = parsed.netloc.lower()
    path = parsed.path.lower()

    features = {
        'url_length': len(url),
        'domain_length': len(domain),
        'path_length': len(path),
        'num_dots': domain.count('.'),
        'num_slashes': path.count('/'),
        'has_https': 1 if url.startswith('https') else 0,
        'has_www': 1 if 'www.' in domain else 0,
        'num_digits': sum(c.isdigit() for c in url),
        'num_special_chars': len(re.findall(r'[^a-zA-Z0-9]', url)),
    }

    return list(features.values())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Load and preprocess dataset
# Assume CSV with 'url' and 'label' columns
# Example dataset: https://www.kaggle.com/datasets/antonyj453/phishing-url-dataset
df = pd.read_csv('/content/drive/MyDrive/Partey/4.phishing.csv')  # Replace with your dataset path
urls = df['Domain'].values
labels = df['Label'].values

In [ ]:
# Tokenize URLs and create vocabulary
tokenized_urls = [tokenize_url(url) for url in urls]
all_tokens = set(token for url_tokens in tokenized_urls for token in url_tokens)
token2idx = {token: idx + 1 for idx, token in enumerate(all_tokens)}  # 0 for padding


In [ ]:
# Convert tokens to sequences
max_len = 100  # Maximum sequence length
sequences = [[token2idx[token] for token in tokens if token in token2idx][:max_len]
             for tokens in tokenized_urls]
sequences = pad_sequences(sequences, maxlen=max_len, padding='post')

In [ ]:
# Extract NLP features
nlp_features = np.array([extract_nlp_features(url) for url in urls])

# Combine tokenized sequences and NLP features
# Normalize NLP features to avoid scale issues
nlp_features = (nlp_features - nlp_features.mean(axis=0)) / (nlp_features.std(axis=0) + 1e-8)
combined_features = np.concatenate([sequences, nlp_features], axis=1)

In [ ]:
# Encode labels
label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(labels)

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    combined_features, encoded_labels, test_size=0.2, random_state=42
)


In [ ]:
# Build LSTM model (inspired by PhiSN's hybrid deep neural network)
vocab_size = len(token2idx) + 1
embedding_dim = 128


In [ ]:
model = Sequential([
    # Embedding layer for tokenized sequences
    Embedding(vocab_size, embedding_dim, input_length=X_train.shape[1]),
    LSTM(64, return_sequences=False),
    Dropout(0.3),
    # Dense layers for combined features
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')  # Binary classification
])


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
# Compile model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


In [ ]:
# Train model
model.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.2, verbose=1)


Epoch 1/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 12s 82ms/step - accuracy: 0.9476 - loss: 0.1973 - val_accuracy: 1.0000 - val_loss: 6.9229e-05
Epoch 2/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 11s 88ms/step - accuracy: 1.0000 - loss: 6.9496e-04 - val_accuracy: 1.0000 - val_loss: 1.0577e-05
Epoch 3/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 10s 85ms/step - accuracy: 1.0000 - loss: 2.1699e-04 - val_accuracy: 1.0000 - val_loss: 2.9286e-06
Epoch 4/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 9s 68ms/step - accuracy: 1.0000 - loss: 1.7174e-04 - val_accuracy: 1.0000 - val_loss: 1.1183e-06
Epoch 5/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 11s 74ms/step - accuracy: 1.0000 - loss: 9.6645e-05 - val_accuracy: 1.0000 - val_loss: 6.0512e-07
Epoch 6/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 11s 87ms/step - accuracy: 1.0000 - loss: 4.8080e-05 - val_accuracy: 1.0000 - val_loss: 3.6514e-07
Epoch 7/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 10s 80ms/step - accuracy: 1.0000 - loss: 5.4302e-05 - val_accuracy: 1.0000 - val_loss: 2.3721e-07
Epoch 8/10
100/100 ━━━━━━━━━━━━━━━━━━━━

In [ ]:
# Evaluate model
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f'Test Accuracy: {accuracy * 100:.2f}%')

Test Accuracy: 100.00%


In [ ]:
# Build LSTM model to mitigate overfitting
model = Sequential([
    Embedding(vocab_size, embedding_dim, input_length=X_train.shape[1]),
    Dropout(0.5), # Added dropout after embedding
    LSTM(64, return_sequences=False),
    Dropout(0.5), # Increased dropout after LSTM
    Dense(32, activation='relu'),
    Dropout(0.5), # Added dropout after first Dense
    Dense(1, activation='sigmoid')  # Binary classification
])

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:

# Compile model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train model
model.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.2, verbose=1)

# Evaluate model
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f'Test Accuracy: {accuracy * 100:.2f}%')

Epoch 1/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 13s 97ms/step - accuracy: 0.9871 - loss: 0.1754 - val_accuracy: 1.0000 - val_loss: 6.9927e-06
Epoch 2/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 10s 95ms/step - accuracy: 1.0000 - loss: 9.9264e-04 - val_accuracy: 1.0000 - val_loss: 7.4387e-07
Epoch 3/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 8s 80ms/step - accuracy: 1.0000 - loss: 3.5756e-04 - val_accuracy: 1.0000 - val_loss: 1.8027e-07
Epoch 4/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 12s 94ms/step - accuracy: 1.0000 - loss: 1.8392e-04 - val_accuracy: 1.0000 - val_loss: 6.5145e-08
Epoch 5/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 10s 95ms/step - accuracy: 1.0000 - loss: 7.5410e-05 - val_accuracy: 1.0000 - val_loss: 3.4246e-08
Epoch 6/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 9s 88ms/step - accuracy: 1.0000 - loss: 6.1021e-05 - val_accuracy: 1.0000 - val_loss: 1.7066e-08
Epoch 7/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 9s 86ms/step - accuracy: 1.0000 - loss: 6.9074e-05 - val_accuracy: 1.0000 - val_loss: 8.2775e-09
Epoch 8/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 1

In [ ]:
# Add final dense layers
final_dense = Dense(16, activation='relu')(merged)
final_dropout = Dropout(0.3)(final_dense)
output_layer = Dense(1, activation='sigmoid')(final_dropout)

NameError: name 'merged' is not defined

# Task
Modify the selected empty cell to implement a Keras Functional API model that accepts two inputs: tokenized sequences and NLP features, and train and evaluate the model.

## Modify the model architecture

### Subtask:
Rebuild the model using the Keras Functional API to accept two inputs: one for the tokenized sequences and one for the NLP features. The tokenized sequences will go through an Embedding and LSTM layer, while the NLP features will go through dense layers directly.


**Reasoning**:
The subtask is to rebuild the model using the Keras Functional API with two inputs. This requires defining input layers, processing each input stream separately with appropriate layers, and then combining them. The instructions outline the specific layers and their configurations for each input stream.



In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, concatenate

# Define input layers
token_input = Input(shape=(max_len,), name='token_input')
nlp_input = Input(shape=(nlp_features.shape[1],), name='nlp_input')

# Process tokenized sequences
embedding_layer = Embedding(vocab_size, embedding_dim)(token_input)
lstm_layer = LSTM(64)(embedding_layer)
lstm_dropout = Dropout(0.3)(lstm_layer)

# Process NLP features
nlp_dense = Dense(32, activation='relu')(nlp_input)
nlp_dropout = Dropout(0.3)(nlp_dense)

In [ ]:
# Prepare input data for the new model
X_train_tokens = X_train[:, :max_len]
X_train_nlp = X_train[:, max_len:]

X_test_tokens = X_test[:, :max_len]
X_test_nlp = X_test[:, max_len:]

NameError: name 'X_train' is not defined

In [ ]:
from tensorflow.keras.layers import concatenate

# Merge the model outputs
merged = concatenate([lstm_dropout, nlp_dropout])

NameError: name 'lstm_dropout' is not defined